# 02 — Building timestamps for each flight

**Question:** can we reconstruct full local timestamps (including arrival date) for every operated flight?

**Hypothesis:** the arrival time we derive from scheduled departure + scheduled elapsed time + time-zone offset must match the reported arrival time (`arr_hhmm`), which we do not use in the calculation.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent
PARQUET = ROOT / "data" / "parquet" / "*.parquet"

con = duckdb.connect()
con.execute(f"CREATE VIEW flights AS SELECT * FROM read_parquet('{PARQUET.as_posix()}')")

In [2]:
con.execute("""
CREATE OR REPLACE VIEW ops AS
WITH base AS (
    -- Operated flights only, with minutes since midnight for the scheduled times
    SELECT
        tail_number, carrier, flight_number, origin, dest, flight_date,
        crs_elapsed, dep_delay, arr_delay, arr_hhmm,
        (CAST(crs_dep_hhmm AS INTEGER) // 100) * 60 + CAST(crs_dep_hhmm AS INTEGER) % 100 AS crs_dep_tod,
        (CAST(crs_arr_hhmm AS INTEGER) // 100) * 60 + CAST(crs_arr_hhmm AS INTEGER) % 100 AS crs_arr_tod
    FROM flights
    WHERE NOT cancelled AND NOT diverted AND tail_number IS NOT NULL
),
tz AS (
    -- Time-zone difference (dest - origin), recovered from the data itself:
    -- scheduled arrival clock time - (scheduled departure clock time + scheduled elapsed time)
    SELECT *,
        ((crs_arr_tod % 1440 - crs_dep_tod % 1440 - crs_elapsed) % 1440 + 1440) % 1440 AS tz_raw
    FROM base
),
tz_signed AS (
    SELECT *, CASE WHEN tz_raw >= 720 THEN tz_raw - 1440 ELSE tz_raw END AS tz_offset_min
    FROM tz
)
SELECT
    tail_number, carrier, flight_number, origin, dest, flight_date, tz_offset_min, arr_hhmm,
    CAST(flight_date AS TIMESTAMP) + to_minutes(crs_dep_tod)                                           AS crs_dep_ts,
    CAST(flight_date AS TIMESTAMP) + to_minutes(crs_dep_tod + dep_delay)                               AS dep_ts,
    CAST(flight_date AS TIMESTAMP) + to_minutes(crs_dep_tod + crs_elapsed + tz_offset_min)             AS crs_arr_ts,
    CAST(flight_date AS TIMESTAMP) + to_minutes(crs_dep_tod + crs_elapsed + tz_offset_min + arr_delay) AS arr_ts,
    dep_delay, arr_delay, crs_elapsed
FROM tz_signed
""")

con.sql("SELECT count(*) AS n_operated_flights FROM ops").df()

,n_operated_flights
0,601570


In [3]:
con.sql("""
    SELECT tz_offset_min / 60 AS tz_hours, count(*) AS n_flights
    FROM ops
    GROUP BY 1
    ORDER BY 1
""").df()

,tz_hours,n_flights
0,-6.000000,113
1,-5.000000,189
2,-4.000000,260
3,-3.000000,23158
4,-2.000000,31211
5,-1.000000,94027
6,0.000000,303559
7,0.166667,1
8,0.666667,1
9,1.000000,94089


In [4]:
con.sql("""
    SELECT
        count(*) AS n_checked,
        sum((((hour(arr_ts) * 60 + minute(arr_ts))
              - ((CAST(arr_hhmm AS INTEGER) // 100) * 60 + CAST(arr_hhmm AS INTEGER) % 100)) % 1440 <> 0)::INT) AS n_mismatch
    FROM ops
""").df()

,n_checked,n_mismatch
0,601570,1.0


**Findings**
- Reconstructed arrival time matches the reported arrival time in 601,569 of 601,570 operated flights.
- Time-zone offsets recovered from the data are whole hours between -6 and +6, except 2 flights (10 and 40 min), most likely errors in the scheduled elapsed time. These will be excluded.
- Timestamps are local to each airport. This is enough to measure turnarounds: the previous arrival and the next departure of an aircraft happen at the same airport, so both use the same clock.